# 🔀 Unified AI API - Validation Tests

## Overview

Validate the Unified AI Wildcard API (`/unified-ai`) across different model providers and API patterns.

This notebook tests:
- **Azure OpenAI Pattern**: `/unified-ai/openai/deployments/{model}/chat/completions`
- **Inference Pattern (Foundry)**: `/unified-ai/models/chat/completions` with Phi-4, DeepSeek-R1
- **Gemini OpenAI Pattern**: `/unified-ai/v1beta/openai/chat/completions`
- **Deployment Discovery**: `GET /unified-ai/openai/deployments` and `GET /unified-ai/openai/deployments/{name}`
- **Authentication Modes**: API Key only, API Key + JWT
- **Model Access Control**: Allowed vs blocked models
- **Load Testing**: Concurrent requests with throttling visualization

> **Prerequisites:** An existing Citadel Governance Hub deployment with the Unified AI API enabled.

### 0️⃣ Initialize Notebook Variables

Configure the following variables according to your environment before running the notebook:

In [ ]:
import os
import sys, json, requests, time
sys.path.insert(1, '../shared')  # add the shared directory to the Python path
import utils
from apimtools import APIMClientTool

inference_api_version = "2024-05-01-preview"
openai_api_version = "2024-02-15-preview"

governance_hub_resource_group = "REPLACE"  ## specify the resource group name where the Governance Hub is located
location = "REPLACE"  ## specify the location of the Governance Hub

# Models to test (REPLACE based on your configured backends and deployed models)
openai_model = "gpt-4o"          # Azure OpenAI model (deployed via backend onboarding)
foundry_model_phi = "Phi-4"       # AI Foundry inference model
foundry_model_deepseek = "DeepSeek-R1"  # AI Foundry reasoning model
gemini_model = "gemini-2.5-flash-lite"  # Gemini model (if Gemini backend configured)

# Set to True if Gemini backend is configured
test_gemini = False

# Set to True if JWT auth is enabled
test_jwt_auth = False

### 1️⃣ Verify Azure CLI and Connected Subscription

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

### 2️⃣ Initialize APIM Client Tool

Discover the Unified AI API and retrieve API key for testing:

In [ ]:
try:
    apimClientTool = APIMClientTool(governance_hub_resource_group)
    apimClientTool.initialize()
    apimClientTool.discover_api("unified-ai")

    apim_resource_gateway_url = str(apimClientTool.apim_resource_gateway_url)
    azure_endpoint = str(apimClientTool.azure_endpoint)

    # Retrieve API key from APIM subscriptions scoped to the unified-ai-product
    api_key = None
    for sub in apimClientTool.apim_subscriptions:
        sub_name = sub.get('name', '')
        # Match subscriptions scoped to the unified-ai product
        if 'unified-ai' in sub_name.lower():
            api_key = sub.get('key')
            utils.print_ok(f"Found API key from subscription: {sub_name}")
            break

    # Fallback: use last available subscription if no unified-ai specific one found
    if not api_key and len(apimClientTool.apim_subscriptions) > 0:
        api_key = apimClientTool.apim_subscriptions[-1].get('key')
        utils.print_info(f"Using fallback subscription: {apimClientTool.apim_subscriptions[-1].get('name')}")

    if not api_key:
        raise Exception("No API key found. Ensure APIM has subscriptions with the unified-ai-product.")

    # Get supported models from the policy fragment
    supported_models = apimClientTool.get_policy_fragment_supported_models("set-backend-pools")
    utils.print_info(f"Supported models: {supported_models}")

    # Build endpoint URLs for different API patterns
    base_url = azure_endpoint.rstrip('/')
    openai_url = f"{base_url}/unified-ai/openai/deployments/{openai_model}/chat/completions?api-version={openai_api_version}"
    inference_url = f"{base_url}/unified-ai/models/chat/completions?api-version={inference_api_version}"
    gemini_url = f"{base_url}/unified-ai/v1beta/openai/chat/completions"
    deployments_url = f"{base_url}/unified-ai/deployments"

    utils.print_info(f"OpenAI endpoint: {openai_url}")
    utils.print_info(f"Inference endpoint: {inference_url}")
    utils.print_info(f"Deployments endpoint: {deployments_url}")

    utils.print_ok(f"Testing tool initialized successfully!")
except Exception as e:
    utils.print_error(f"Error initializing APIM Client Tool: {e}")

---
## Test 1 — Azure OpenAI Pattern

Route through the OpenAI-compatible path: `POST /unified-ai/openai/deployments/{model}/chat/completions`

In [ ]:
utils.print_info(f"Testing Azure OpenAI pattern with model: {openai_model}")

messages = {
    "messages": [
        {"role": "system", "content": "You are a helpful assistant. Keep responses brief."},
        {"role": "user", "content": "What is 2+2? Answer in one word."}
    ]
}

try:
    response = requests.post(
        openai_url,
        headers={'api-key': api_key},
        json=messages,
        timeout=30
    )

    utils.print_response_code(response)

    if response.status_code == 200:
        data = json.loads(response.text)
        content = data.get("choices", [{}])[0].get("message", {}).get("content", "")
        utils.print_ok(f"💬 Response: {content}")
        utils.print_info(f"   Region: {response.headers.get('x-ms-region', 'N/A')}")
        utils.print_info(f"   Backend: {response.headers.get('UAIG-Backend', 'N/A')}")
        utils.print_info(f"   API Type: {response.headers.get('UAIG-API-Type', 'N/A')}")
    else:
        utils.print_error(f"Error: {response.text[:300]}")
except Exception as e:
    utils.print_error(f"Request failed: {e}")

## Test 2 — Inference Pattern (Foundry Models)

Route through the inference path: `POST /unified-ai/models/chat/completions`

Testing with Phi-4 and DeepSeek-R1 models deployed via AI Foundry.

In [ ]:
# Test Phi-4
utils.print_info(f"Testing Foundry Inference with model: {foundry_model_phi}")

phi_messages = {
    "model": foundry_model_phi,
    "messages": [
        {"role": "system", "content": "You are a helpful assistant. Keep responses brief."},
        {"role": "user", "content": "What is the capital of France? Answer in one word."}
    ]
}

try:
    response = requests.post(
        inference_url,
        headers={'api-key': api_key},
        json=phi_messages,
        timeout=60
    )

    utils.print_response_code(response)

    if response.status_code == 200:
        data = json.loads(response.text)
        content = data.get("choices", [{}])[0].get("message", {}).get("content", "")
        utils.print_ok(f"💬 Phi-4 Response: {content[:200]}")
        utils.print_info(f"   Backend: {response.headers.get('UAIG-Backend', 'N/A')}")
        utils.print_info(f"   API Type: {response.headers.get('UAIG-API-Type', 'N/A')}")
    else:
        utils.print_error(f"Error: {response.text[:300]}")
except Exception as e:
    utils.print_error(f"Request failed: {e}")

In [ ]:
# Test DeepSeek-R1
utils.print_info(f"Testing Foundry Inference with model: {foundry_model_deepseek}")

deepseek_messages = {
    "model": foundry_model_deepseek,
    "messages": [
        {"role": "user", "content": "What is 15 * 23? Think step by step."}
    ]
}

try:
    response = requests.post(
        inference_url,
        headers={'api-key': api_key},
        json=deepseek_messages,
        timeout=120
    )

    utils.print_response_code(response)

    if response.status_code == 200:
        data = json.loads(response.text)
        content = data.get("choices", [{}])[0].get("message", {}).get("content", "")
        utils.print_ok(f"💬 DeepSeek-R1 Response: {content[:300]}")
        utils.print_info(f"   Backend: {response.headers.get('UAIG-Backend', 'N/A')}")
    else:
        utils.print_error(f"Error: {response.text[:300]}")
except Exception as e:
    utils.print_error(f"Request failed: {e}")

## Test 3 — Deployment Discovery

Validate deployment listing through `GET /unified-ai/openai/deployments`

In [ ]:
# List all deployments
utils.print_info("Listing all available deployments...")

try:
    response = requests.get(
        deployments_url,
        headers={'api-key': api_key},
        timeout=30
    )

    utils.print_response_code(response)

    if response.status_code == 200:
        data = json.loads(response.text)
        deployments = data.get("value", [])
        utils.print_ok(f"Found {len(deployments)} deployments:")
        for d in deployments:
            name = d.get('name', 'unknown')
            model_format = d.get('properties', {}).get('model', {}).get('format', 'N/A')
            utils.print_info(f"  • {name} (format: {model_format})")
    else:
        utils.print_error(f"Error: {response.text[:300]}")
except Exception as e:
    utils.print_error(f"Request failed: {e}")

In [ ]:
# Get specific deployment (should return 200)
utils.print_info(f"Getting deployment: {openai_model}")

try:
    response = requests.get(
        f"{deployments_url}/{openai_model}",
        headers={'api-key': api_key},
        timeout=30
    )

    utils.print_response_code(response)
    if response.status_code == 200:
        data = json.loads(response.text)
        utils.print_ok(f"Found deployment: {data.get('name', 'unknown')}")
    else:
        utils.print_error(f"Unexpected: {response.text[:200]}")
except Exception as e:
    utils.print_error(f"Request failed: {e}")

# Get non-existent deployment (should return 404)
utils.print_info(f"Getting non-existent deployment: nonexistent-model-xyz")

try:
    response = requests.get(
        f"{deployments_url}/nonexistent-model-xyz",
        headers={'api-key': api_key},
        timeout=30
    )

    if response.status_code == 404:
        utils.print_ok(f"Correctly returned 404 for non-existent model")
    else:
        utils.print_error(f"Unexpected status {response.status_code}: {response.text[:200]}")
except Exception as e:
    utils.print_error(f"Request failed: {e}")

## Test 4 — Gemini OpenAI Pattern (if configured)

Route through the Gemini-compatible path: `POST /unified-ai/v1beta/openai/chat/completions`

In [ ]:
if test_gemini:
    utils.print_info(f"Testing Gemini OpenAI pattern with model: {gemini_model}")

    gemini_messages = {
        "model": gemini_model,
        "messages": [
            {"role": "user", "content": "What is the speed of light? Answer briefly."}
        ]
    }

    try:
        response = requests.post(
            gemini_url,
            headers={'api-key': api_key},
            json=gemini_messages,
            timeout=30
        )

        utils.print_response_code(response)

        if response.status_code == 200:
            data = json.loads(response.text)
            content = data.get("choices", [{}])[0].get("message", {}).get("content", "")
            utils.print_ok(f"💬 Gemini Response: {content[:200]}")
            utils.print_info(f"   Backend: {response.headers.get('UAIG-Backend', 'N/A')}")
            utils.print_info(f"   API Type: {response.headers.get('UAIG-API-Type', 'N/A')}")
        else:
            utils.print_error(f"Error: {response.text[:300]}")
    except Exception as e:
        utils.print_error(f"Request failed: {e}")
else:
    utils.print_info("Skipping Gemini test (test_gemini = False)")

## Test 5 — Authentication Modes

Validate that API Key authentication works and missing keys are rejected:

In [ ]:
# Test: Valid API Key -> should succeed
utils.print_info("Testing with valid API key...")

response = requests.post(
    openai_url,
    headers={'api-key': api_key},
    json={"messages": [{"role": "user", "content": "Hi"}]},
    timeout=30
)
if response.status_code == 200:
    utils.print_ok(f"✅ Valid API key: {response.status_code} (expected 200)")
else:
    utils.print_error(f"❌ Valid API key: {response.status_code} (expected 200)")

# Test: Missing API Key -> should return 401
utils.print_info("Testing without API key...")

response = requests.post(
    openai_url,
    headers={},
    json={"messages": [{"role": "user", "content": "Hi"}]},
    timeout=30
)
if response.status_code == 401:
    utils.print_ok(f"✅ Missing API key: {response.status_code} (expected 401)")
else:
    utils.print_error(f"❌ Missing API key: {response.status_code} (expected 401)")

## Test 6 — Load Test & Throttling

Send multiple requests over 30 seconds to observe rate limiting behavior:

In [ ]:
test_duration = 30
api_runs = []

test_model = openai_model
test_url = openai_url
utils.print_info(f"Load testing with model: {test_model} for {test_duration} seconds")

messages = {
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Count from 1 to 5."}
    ]
}

start_time = time.time()
run_count = 0

while (time.time() - start_time) < test_duration:
    run_count += 1
    call_start = time.time()

    try:
        response = requests.post(
            test_url,
            headers={'api-key': api_key},
            json=messages,
            timeout=30
        )

        elapsed = time.time() - start_time
        total_tokens = 0
        if response.status_code == 200:
            data = json.loads(response.text)
            total_tokens = data.get("usage", {}).get("total_tokens", 0)

        api_runs.append((call_start, total_tokens, response.status_code, elapsed))
    except Exception as e:
        api_runs.append((call_start, 0, 500, time.time() - start_time))

    time.sleep(0.2)

success = sum(1 for r in api_runs if r[2] == 200)
throttled = sum(1 for r in api_runs if r[2] == 429)
errors = sum(1 for r in api_runs if r[2] not in [200, 429])

utils.print_ok(f"Load test complete: {len(api_runs)} calls")
utils.print_info(f"  ✅ Success: {success} | ⛔ Throttled: {throttled} | ❌ Errors: {errors}")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

if api_runs:
    fig, ax = plt.subplots(1, 1, figsize=(14, 5))

    times = [r[3] for r in api_runs]
    tokens = [r[1] for r in api_runs]
    status_codes = [r[2] for r in api_runs]

    colors = [
        'tab:green' if code == 200
        else 'tab:red' if code == 429
        else 'tab:orange'
        for code in status_codes
    ]

    ax.bar(times, tokens, color=colors, width=0.3, alpha=0.7)

    throttled_times = [t for t, code in zip(times, status_codes) if code == 429]
    if throttled_times:
        max_tokens = max(tokens) if tokens else 1
        ax.scatter(throttled_times, [max_tokens * 0.05] * len(throttled_times),
                  marker='x', s=50, color='darkred', zorder=5)

    total_tokens = sum(tokens)
    stats_text = f"Total: {len(api_runs)} calls | Success: {success} | Throttled: {throttled} | Tokens: {total_tokens}"
    ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, fontsize=9,
           verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    ax.set_title('Unified AI API - Load Test Results', fontsize=12, fontweight='bold')
    ax.set_xlabel('Time (seconds)')
    ax.set_ylabel('Tokens per call')

    legend_items = [
        Patch(facecolor='tab:green', alpha=0.7, label='Success (200)'),
        Patch(facecolor='tab:red', alpha=0.7, label='Throttled (429)'),
        Patch(facecolor='tab:orange', alpha=0.7, label='Error'),
    ]
    ax.legend(handles=legend_items, loc='upper right')

    plt.tight_layout()
    plt.show()
else:
    print('No data to visualize. Run the load test first.')

## Summary

| Test | Pattern | Expected |
|------|---------|----------|
| Azure OpenAI | `POST /unified-ai/openai/deployments/{model}/chat/completions` | 200 |
| Foundry Phi-4 | `POST /unified-ai/models/chat/completions` (body: model=Phi-4) | 200 |
| Foundry DeepSeek-R1 | `POST /unified-ai/models/chat/completions` (body: model=DeepSeek-R1) | 200 |
| List Deployments | `GET /unified-ai/openai/deployments` | 200 with model list |
| Get Deployment | `GET /unified-ai/openai/deployments/{name}` | 200 or 404 |
| Gemini | `POST /unified-ai/v1beta/openai/chat/completions` | 200 (if configured) |
| Valid API Key | Request with api-key header | 200 |
| Missing API Key | Request without api-key header | 401 |
| Load Test | 30s burst requests | Mix of 200/429 |